# Quantum error correction below threshold

*quantum · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/quantum/error_correction_threshold)

A bit-flip repetition code spreads one logical bit over $d$ noisy qubits and keeps measuring their parities. Thousands of Monte Carlo shots run as one $(S, R, d)$ tensor, an exact minimum-weight decoder reads the space–time syndrome history, and the curves for $d = 3, 5, 7, 9$ cross at the threshold: below it every step $d \to d + 2$ divides the logical error rate by a factor $\Lambda > 1$, the quantity behind Google's 2024 below-threshold surface-code memory.

**Vector form**

$$\sigma_t = H\,x_t \oplus m_t,\qquad P_L = \sum_{k > d/2}\binom{d}{k}p^k(1-p)^{d-k}\ \text{(no readout errors)},\qquad \Lambda = \frac{\varepsilon_d}{\varepsilon_{d+2}}$$

**Index (tensor) form**

$$x_{sti} = \bigoplus_{t'\le t} e_{st'i},\qquad \sigma_{stj} = H_{ji}\,x_{sti} + m_{stj}\ (\mathrm{mod}\ 2),\qquad D_{stj} = \sigma_{stj}\oplus\sigma_{s,t-1,j},\qquad C_t(x) = \min_y\big[C_{t-1}(y) + w\,\lvert x\oplus y\rvert\big] + w_m K_{\sigma_t x}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $e_{sti},\ m_{stj}$ | `sample_noise(S, R, d, p, q, gen, dev)` | (S, R, d), (S, R, d-1) | data flips (probability $p$) and misread checks (probability $q$) for shot $s$, round $t$ |
| $x_{sti}$ | `torch.cumsum(e, dim=1) % 2` | (S, R, d) | accumulated error on data qubit $i$ after round $t$ |
| $H_{ji}$ | `parity_check(d)` | (d-1, d) | check $j$ compares neighbours $i = j$ and $i = j + 1$ ($Z_jZ_{j+1}$) |
| $H_{ji}\,x_{sti}$ | `einsum('ji,sti->stj', h, x) % 2` | (S, R, d-1) | repeated $i$ ⇒ summed: the parity each check should report |
| $D_{stj}$ | `detection_events(sigma)` | (S, R, d-1) | a syndrome bit that changed: the end of an error chain in space–time |
| $C_t(x)$ | `relax_data_errors(cost, w, d) + w_m * K[sigma_t]` | (S, 2^d) | cheapest history ending in error pattern $x$, $w = \ln\frac{1-p}{p}$; the min over $y$ is $d$ one-bit flips |
| $\sum_i x_{si} > d/2$ | `einsum('si->s', x) > d / 2` | (S,) | majority vote: the optimal decoder when checks never misread |
| $\Lambda_{d/d+2}$ | `eps[:-1] / eps[1:]` | (D-1, P) | error suppression per round, $1 - 2P_L = (1 - 2\varepsilon)^R$ |

The code stores $\lvert 0_L\rangle = \lvert 00\ldots0\rangle$ and $\lvert 1_L\rangle = \lvert 11\ldots1\rangle$; the parity checks $Z_jZ_{j+1}$ reveal where neighbours disagree without revealing the logical value. In the code-capacity model (one round, perfect checks) the best decoder is a majority vote and the simulated $P_L$ follows the exact binomial tail (dashed), with a threshold of exactly $p = 1/2$. In the phenomenological model the checks also misread, so a single syndrome cannot be trusted; the decoder looks for the fewest data flips plus misreads that explain the whole $d$-round history. That is the problem minimum-weight perfect matching solves; in 1D it is solved here exactly by dynamic programming over the $2^d$ error patterns (so $d \le 11$). For this model the threshold is near $p \approx 10\%$ (about 10.3 % for very large $d$), and the small-$d$ crossings land close to it when each code is run for $d$ rounds. Below threshold larger codes win, and $\Lambda = \varepsilon_d/\varepsilon_{d+2}$ grows as $p$ falls (roughly as $p_{th}/p$); above it, adding qubits only adds more places to fail. Research context: Google Quantum AI and Collaborators, Quantum error correction below the surface code threshold, Nature 638, 920 (2025; arXiv:2408.13687, 2024) ran distance-3, 5 and 7 surface-code memories on their Willow processor and reported $\Lambda \approx 2.1$, with a distance-7 logical error of roughly 0.14 % per cycle (approximate values). Earlier steps were repetition codes (Nature 595, 383, 2021) and a distance-5 surface code slightly beating distance 3 (Nature 614, 676, 2023). Repetition code vs surface code: the repetition code is a 1D line that corrects only bit flips ($X$ errors) — a phase flip ($Z$) on any qubit is a logical error, so it is not a full quantum memory. The surface code is a 2D lattice with both $Z$-type and $X$-type checks and corrects both; its decoding graph is 3D (two space dimensions plus time) and its thresholds are lower, about 3 % for this phenomenological noise and roughly 1 % for realistic circuit noise (approximate). Simplifications: independent errors, a perfect final readout, and no leakage or correlated bursts, which in real devices set a floor on the logical error rate.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/quantum/error_correction_threshold/main.py
"""Main loop: noise model → batched Monte Carlo shots → decoder → threshold and Λ → plot."""

import math
from pathlib import Path

from problems._common.runtime import cli, device, fast_mode

from . import plot, solve, tensors
from .equation import binomial_logical_error


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    fast = fast_mode()
    model = params["model"]
    ds = tensors.distances(5 if fast else params["d_max"])
    ps = tensors.error_rates(5 if fast else params["n_p"], params["p_min"], params["p_max"])
    shots = 1000 if fast else params["shots"]
    rounds = "1 round, perfect checks" if model == "code_capacity" else "d rounds, checks misread with q = p·ratio"
    print(f"bit-flip repetition code, d = {ds}, {shots} shots per point, {model} noise ({rounds})")

    result = solve.sweep(ds, ps, params["q_ratio"], model, shots, seed=0, device=dev)
    pl = result["p_logical"]

    print("\n   p     " + "".join(f"  P_L(d={d})" for d in ds))
    for b, p in enumerate(ps.tolist()):
        print(f"{p:7.4f}  " + "".join(f"  {pl[a, b]:.3e}  " for a in range(len(ds))))
    if model == "code_capacity":
        worst = max(abs(pl[a] - binomial_logical_error(ps, d)).max().item() for a, d in enumerate(ds))
        print(f"majority vote vs exact binomial tail: max |difference| = {worst:.1e} (statistical error ~1/√shots)")

    crossings = [solve.crossing(ps, pl[a], pl[a + 1]) for a in range(len(ds) - 1)]
    for d, pc in zip(ds, crossings):
        print(
            f"curves d = {d} and d = {d + 2} cross at p ≈ {pc:.3f}"
            if math.isfinite(pc)
            else f"d = {d}/{d + 2}: no crossing in range"
        )
    finite = [c for c in crossings if math.isfinite(c)]
    p_th = sum(finite) / len(finite) if finite else float("nan")
    if model == "code_capacity" and not finite:
        print("code capacity: the majority vote threshold is exactly p = 0.5 (raise p_max to see it)")

    lam = solve.suppression(result["eps"], pl, shots)
    for k, d in enumerate(ds[:-1]):
        good = [(p, lam[k, b].item()) for b, p in enumerate(ps.tolist()) if lam[k, b] == lam[k, b]]
        good = [(p, v) for p, v in good if not math.isfinite(p_th) or p < p_th]
        if good:
            text = ", ".join(f"{v:.2f} at p = {p:.3f}" for p, v in good[:3])
            print(f"Λ_{d}/{d + 2} = ε_{d}/ε_{d + 2} (per round): {text}")
    print("below threshold Λ > 1: every +2 in distance divides the logical error rate by Λ")
    return plot.animate(ds, ps, result, lam, p_th, model, out_dir / "error_correction_threshold.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/quantum/error_correction_threshold/equation.py
r"""Physics → tensor translation for the bit-flip repetition code.

A distance-d repetition code stores one logical bit in d data qubits (|0_L⟩ = |00…0⟩). Each round t,
every data qubit flips with probability p, and the d − 1 parity checks Z_i Z_{i+1} are measured,
each reading wrong with probability q. With s = shot, t = round, i = data qubit, j = check:

    x_{sti} = ⊕_{t' ≤ t} e_{st'i}                      accumulated flips (cumulative sum mod 2)
    σ_{stj} = H_{ji} x_{sti} + m_{stj}   (mod 2)          i summed: measured syndrome
    D_{stj} = σ_{stj} ⊕ σ_{s,t-1,j}                     detection events (what the decoder looks at)

H_{ji} is the (d − 1) × d parity-check matrix, H_{j,j} = H_{j,j+1} = 1. Without measurement errors the
best decoder is a majority vote, and the logical error probability is the binomial tail

    P_L = Σ_{k > d/2} C(d, k) p^k (1 − p)^{d − k}.

Per-round logical error ε from P_L after R rounds:  1 − 2 P_L = (1 − 2ε)^R.
Error-suppression factor:  Λ_{d/d+2} = ε_d / ε_{d+2}.
"""

import math

import torch


def parity_check(d: int, device: torch.device | None = None) -> torch.Tensor:
    """H_{ji}: (d − 1, d) with ones on the diagonal and the superdiagonal."""
    h = torch.zeros(d - 1, d, dtype=torch.long, device=device)
    j = torch.arange(d - 1, device=device)
    h[j, j] = 1
    h[j, j + 1] = 1
    return h


def accumulate(flips: torch.Tensor) -> torch.Tensor:
    r"""x_{sti} = ⊕_{t' ≤ t} e_{st'i}.   flips (S, R, d) bool → (S, R, d) long in {0, 1}."""
    return torch.cumsum(flips.long(), dim=1) % 2


def syndrome(x: torch.Tensor, h: torch.Tensor, meas_flips: torch.Tensor | None = None) -> torch.Tensor:
    r"""σ_{stj} = H_{ji} x_{sti} + m_{stj} (mod 2) — repeated i summed.   (S, R, d) → (S, R, d − 1)."""
    s = torch.einsum("ji,sti->stj", h, x) % 2
    if meas_flips is not None:
        s = (s + meas_flips.long()) % 2
    return s


def detection_events(s: torch.Tensor) -> torch.Tensor:
    r"""D_{stj} = σ_{stj} ⊕ σ_{s,t-1,j} with σ_{s,0,j} = 0.   (S, R, d − 1) → (S, R, d − 1)."""
    previous = torch.cat([torch.zeros_like(s[:, :1]), s[:, :-1]], dim=1)
    return (s + previous) % 2


def majority_vote_fails(x_final: torch.Tensor) -> torch.Tensor:
    r"""Logical error ⇔ more than half the data qubits are flipped: Σ_i x_{si} > d/2.   (S, d) → (S,) bool."""
    d = x_final.shape[-1]
    return torch.einsum("si->s", x_final) > d / 2


def binomial_logical_error(p: torch.Tensor, d: int) -> torch.Tensor:
    r"""Exact majority-vote failure: P_L = Σ_{k > d/2} C(d, k) p^k (1 − p)^{d − k}."""
    p = torch.as_tensor(p, dtype=torch.float64)
    return sum(math.comb(d, k) * p**k * (1 - p) ** (d - k) for k in range(d // 2 + 1, d + 1))


def per_round(p_logical: torch.Tensor, rounds: int) -> torch.Tensor:
    r"""ε from 1 − 2 P_L = (1 − 2ε)^R (an error per round; for R = 1, ε = P_L)."""
    x = (1 - 2 * p_logical).clamp(min=0.0)
    return (1 - x ** (1.0 / rounds)) / 2


def log_weight(p: float) -> float:
    """Matching weight of one error with probability p: w = ln((1 − p)/p) (minus its log-likelihood ratio)."""
    p = min(max(p, 1e-12), 0.5 - 1e-12)
    return math.log((1 - p) / p)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/quantum/error_correction_threshold/tensors.py
"""Build the concrete tensors: the noise samples (all shots at once) and the decoder's lookup tables."""

import torch


def distances(d_max: int) -> list[int]:
    """Odd code distances 3, 5, …, d_max."""
    return list(range(3, d_max + 1, 2))


def error_rates(n: int, p_min: float, p_max: float) -> torch.Tensor:
    """n physical error rates p, log-spaced: (P,)."""
    return torch.logspace(torch.log10(torch.tensor(p_min)), torch.log10(torch.tensor(p_max)), n, dtype=torch.float64)


def sample_noise(
    shots: int, rounds: int, d: int, p: float, q: float, gen: torch.Generator, device: torch.device
) -> tuple[torch.Tensor, torch.Tensor]:
    """Independent errors for every shot in one tensor.

    e_{sti}: (S, R, d) bool, data qubit i flips in round t with probability p
    m_{stj}: (S, R, d − 1) bool, check j reads wrong in round t with probability q
    """
    e = torch.rand(shots, rounds, d, generator=gen).to(device) < p
    m = torch.rand(shots, rounds, d - 1, generator=gen).to(device) < q
    return e, m


def bit_table(d: int, device: torch.device | None = None) -> torch.Tensor:
    """bits[x, i] = bit i of the integer x: every one of the 2^d error patterns, (2^d, d) long."""
    x = torch.arange(2**d, device=device)
    return (x[:, None] >> torch.arange(d, device=device)) & 1


def pack_bits(s: torch.Tensor) -> torch.Tensor:
    """Pack a bit string (…, n) of 0/1 (a syndrome or an error pattern) into one integer Σ_j s_j 2^j: (…,)."""
    powers = 2 ** torch.arange(s.shape[-1], device=s.device)
    return torch.einsum("...j,j->...", s.long(), powers)


def mismatch_table(d: int, device: torch.device | None = None) -> torch.Tensor:
    """K[σ, x] = number of checks where the observed syndrome σ disagrees with the syndrome of pattern x.

    Shape (2^{d−1}, 2^d). K[σ, x] measurement errors explain seeing σ while the data error is x.
    """
    bits = bit_table(d, device)  # (2^d, d)
    syn = (bits[:, :-1] + bits[:, 1:]) % 2  # (2^d, d − 1) syndrome of each pattern
    obs = bit_table(d - 1, device)  # (2^{d−1}, d − 1) every possible observed syndrome
    return (obs[:, None, :] != syn[None, :, :]).sum(-1)  # (2^{d−1}, 2^d)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/quantum/error_correction_threshold/solve.py
"""Monte Carlo memory experiment + exact minimum-weight decoder on the space–time syndrome history.

Decoder. The unknown is the accumulated error pattern x_t ∈ {0, 1}^d after each round. A guess for the
whole history costs w_data per data flip (x_t differs from x_{t−1}) plus w_meas per check whose reading
disagrees with H x_t. The cheapest history is the minimum-weight explanation of the syndromes — the
same answer minimum-weight perfect matching gives. Because the code is 1D, it can be found exactly by
dynamic programming over time (Viterbi) on a (shots, 2^d) cost tensor:

    C_t(x) = min_y [C_{t−1}(y) + w_data · |x ⊕ y|] + w_meas · K[σ_t, x]

The min over y factorises over qubits, so it is d cheap "flip one bit" relaxations.
At the end the data qubits are read out, which fixes x_R up to a global flip (the logical operator);
the decoder fails when the flipped pattern is cheaper than the true one.
"""

import math

import torch

from problems._common.progress import steps

from . import tensors
from .equation import accumulate, log_weight, majority_vote_fails, parity_check, per_round, syndrome


def relax_data_errors(cost: torch.Tensor, w: float, d: int) -> torch.Tensor:
    """min over y of cost(y) + w·|x ⊕ y|, one qubit at a time.   cost (S, 2^d) → (S, 2^d)."""
    s = cost.shape[0]
    for i in range(d):
        c = cost.view(s, 2 ** (d - 1 - i), 2, 2**i)  # axis 2 is bit i of the pattern index
        flipped = c.flip(2).add_(w)  # cost of reaching x by flipping qubit i
        cost = torch.minimum(c, flipped, out=flipped).view(s, -1)  # in place: keeps memory traffic low
    return cost


def decode_fails(
    x: torch.Tensor, sigma: torch.Tensor, mismatch: torch.Tensor, w_data: float, w_meas: float
) -> torch.Tensor:
    """Run the space–time decoder. x (S, R, d) true errors, sigma (S, R, d − 1) measured syndromes.

    Returns (S,) float: 1 if the decoder picks the wrong logical class, ½ on an exact tie.
    """
    n_shots, rounds, d = x.shape
    cost = torch.full((n_shots, 2**d), 1e6, device=x.device)
    cost[:, 0] = 0.0  # the code starts in |0_L⟩ with no errors
    sigma_idx = tensors.pack_bits(sigma)  # (S, R)
    for t in range(rounds):
        cost = relax_data_errors(cost, w_data, d)
        cost = cost + w_meas * mismatch[sigma_idx[:, t]]  # (S, 2^d)
    true_idx = tensors.pack_bits(x[:, -1])  # (S,) index of the true final error pattern
    flipped_idx = true_idx ^ (2**d - 1)  # same syndrome, opposite logical value
    c_true = cost.gather(1, true_idx[:, None])[:, 0]
    c_flip = cost.gather(1, flipped_idx[:, None])[:, 0]
    return (c_flip < c_true).float() + 0.5 * (c_flip == c_true).float()


def run_point(
    d: int, p: float, q: float, rounds: int, shots: int, gen: torch.Generator, device: torch.device, chunk: int = 8192
) -> tuple[float, torch.Tensor]:
    """Logical error probability after `rounds` rounds and one sample detection-event history (R, d − 1).

    q = 0 and rounds = 1 is the code-capacity model, decoded by a majority vote.
    """
    h = parity_check(d, device)
    mismatch = tensors.mismatch_table(d, device).float()
    w_data, w_meas = log_weight(p), log_weight(q)
    failures, sample = 0.0, None
    for start in range(0, shots, chunk):
        n = min(chunk, shots - start)
        e, m = tensors.sample_noise(n, rounds, d, p, q, gen, device)
        x = accumulate(e)  # (n, R, d)
        if q == 0.0:
            failures += majority_vote_fails(x[:, -1]).float().sum().item()
        else:
            sigma = syndrome(x, h, m)  # (n, R, d − 1)
            failures += decode_fails(x, sigma, mismatch, w_data, w_meas).sum().item()
        if sample is None:
            sample = syndrome(x[:1], h, m[:1])[0]
    return failures / shots, sample


def sweep(
    ds: list[int], ps: torch.Tensor, q_ratio: float, model: str, shots: int, seed: int, device: torch.device
) -> dict:
    """Logical error rates for every distance and physical error rate.

    Returns P_L (D, P), its binomial standard error, the per-round ε, and sample syndrome histories.
    """
    gen = torch.Generator().manual_seed(seed)
    p_logical = torch.zeros(len(ds), len(ps), dtype=torch.float64)
    samples = {}
    jobs = [(a, d, b, p) for a, d in enumerate(ds) for b, p in enumerate(ps.tolist())]
    for k in steps(len(jobs), "(d, p) points"):
        a, d, b, p = jobs[k]
        rounds = 1 if model == "code_capacity" else d
        q = 0.0 if model == "code_capacity" else min(0.5, q_ratio * p)
        p_logical[a, b], samples[(d, b)] = run_point(d, p, q, rounds, shots, gen, device)
    stderr = torch.sqrt(p_logical * (1 - p_logical) / shots).clamp(min=1.0 / shots)
    rounds = torch.tensor([1 if model == "code_capacity" else d for d in ds], dtype=torch.float64)
    eps = torch.stack([per_round(p_logical[a], int(rounds[a])) for a in range(len(ds))])
    return {"p_logical": p_logical, "stderr": stderr, "eps": eps, "samples": samples, "shots": shots}


def crossing(ps: torch.Tensor, eps_small: torch.Tensor, eps_large: torch.Tensor) -> float:
    """Physical error rate where the larger code stops helping (log–log interpolation), nan if none."""
    ok = (eps_small > 0) & (eps_large > 0)
    lp, diff = torch.log(ps[ok]), torch.log(eps_small[ok]) - torch.log(eps_large[ok])
    for k in range(len(diff) - 1):
        if diff[k] > 0 >= diff[k + 1]:
            f = diff[k] / (diff[k] - diff[k + 1])
            return math.exp(lp[k] + f * (lp[k + 1] - lp[k]))
    return float("nan")


def suppression(eps: torch.Tensor, p_logical: torch.Tensor, shots: int, min_failures: int = 10) -> torch.Tensor:
    """Λ_{d/d+2} = ε_d / ε_{d+2} for consecutive distances; nan where too few failures were seen.  (D − 1, P)"""
    enough = p_logical * shots >= min_failures
    lam = eps[:-1] / eps[1:]
    return torch.where(enough[:-1] & enough[1:], lam, torch.full_like(lam, float("nan")))


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/quantum/error_correction_threshold/plot.py
"""Sweep the physical error rate: a sample syndrome history, the threshold plot and the suppression factor Λ."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.colors import ListedColormap
from matplotlib.ticker import NullFormatter

from problems._common.plotting import frame_indices, save_gif

from .equation import binomial_logical_error, detection_events


def animate(
    ds: list[int],
    ps: torch.Tensor,
    result: dict,
    lam: torch.Tensor,
    p_th: float,
    model: str,
    path: Path,
) -> Path:
    p = ps.numpy()
    pl, err = result["p_logical"].numpy(), result["stderr"].numpy()  # (D, P)
    lam_np = lam.numpy().copy()  # (D − 1, P)
    if np.isfinite(p_th):
        lam_np[:, p >= p_th] = np.nan  # per-round Λ is only meaningful below threshold
    d_max = ds[-1]

    fig, (ax_s, ax_p, ax_l) = plt.subplots(1, 3, figsize=(11, 3.8), gridspec_kw={"width_ratios": [0.8, 1.2, 1.0]})

    # (a) one shot of the largest code: detection events in space–time
    first = detection_events(result["samples"][(d_max, 0)][None])[0].numpy()
    image = ax_s.imshow(
        first, cmap=ListedColormap(["#eff6ff", "#dc2626"]), vmin=0, vmax=1, aspect="auto", origin="lower"
    )
    ax_s.set_xlabel("check j")
    ax_s.set_ylabel("round t")
    ax_s.grid(False)
    title_s = ax_s.set_title("d, p: detection events", fontsize=9)

    # (b) logical error probability of a d-round memory experiment
    lines, bars = [], []
    for d in ds:
        (line,) = ax_p.plot([], [], "o-", ms=4, lw=1.6, label=f"d = {d}")
        lines.append(line)
        if model == "code_capacity":
            exact = binomial_logical_error(ps, d).numpy()
            ax_p.plot(p, exact, "--", lw=1, color=line.get_color(), alpha=0.7)
        bars.append(line.get_color())
    if np.isfinite(p_th):
        ax_p.axvline(p_th, color="#f59e0b", lw=1.5, ls="--")
        ax_p.text(p_th, 1.0 / result["shots"], f"$p_{{th}}$ ≈ {p_th:.3f} ", color="#b45309", fontsize=8, ha="right")
    ax_p.set_xscale("log")
    ax_p.set_yscale("log")
    ax_p.set_xlim(p[0] * 0.9, p[-1] * 1.1)
    ax_p.set_ylim(0.5 / result["shots"], 0.7)
    ax_p.set_xlabel("physical error rate p")
    ax_p.set_ylabel("logical error $P_L$" + (" (dashed: exact)" if model == "code_capacity" else " per d rounds"))
    ax_p.legend(loc="lower right", fontsize=7)
    shade = [None]

    # (c) error suppression Λ = ε_d / ε_{d+2}
    lam_lines = [ax_l.plot([], [], "s-", ms=4, lw=1.4, label=f"{d}→{d + 2}")[0] for d in ds[:-1]]
    ax_l.axhline(1.0, color="#9ca3af", lw=1.5)
    ax_l.set_xscale("log")
    ax_l.set_xlim(p[0] * 0.9, p[-1] * 1.1)
    finite = lam_np[np.isfinite(lam_np)]
    ax_l.set_ylim(0, max(2.5, float(finite.max()) * 1.1) if finite.size else 3)
    ax_l.set_xlabel("physical error rate p")
    ax_l.set_ylabel(r"$\Lambda = \varepsilon_d / \varepsilon_{d+2}$ (per round)")
    ax_l.legend(loc="upper right", fontsize=7, title="d")
    if np.isfinite(p_th):
        ax_l.axvspan(p_th, p[-1] * 1.1, color="#f3f4f6", zorder=0)
        ax_l.text(p_th, 0.15, " above threshold", fontsize=7, color="#6b7280")

    for ax in (ax_p, ax_l):
        ticks = [v for v in (0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5) if p[0] * 0.9 <= v <= p[-1] * 1.1]
        ax.set_xticks(ticks, [f"{v:g}" for v in ticks])
        ax.xaxis.set_minor_formatter(NullFormatter())

    def update(b: int):
        events = detection_events(result["samples"][(d_max, b)][None])[0].numpy()
        image.set_data(events)
        title_s.set_text(f"d = {d_max}, p = {p[b]:.3f}: {int(events.sum())} detection events")
        for a, line in enumerate(lines):
            keep = pl[a, : b + 1] > 0
            line.set_data(p[: b + 1][keep], pl[a, : b + 1][keep])
        if shade[0] is not None:
            shade[0].remove()
        lo = np.clip(pl[:, : b + 1] - err[:, : b + 1], 1e-9, None)
        shade[0] = ax_p.vlines(
            np.broadcast_to(p[: b + 1], lo.shape).ravel(),
            lo.ravel(),
            (pl[:, : b + 1] + err[:, : b + 1]).ravel(),
            colors=np.repeat(bars, b + 1),
            lw=1,
        )
        for k, line in enumerate(lam_lines):
            line.set_data(p[: b + 1], lam_np[k, : b + 1])

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(p), len(p)), path, fps=2)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'model': 'phenomenological',
    'd_max': 9,
    'shots': 10000,
    'n_p': 10,
    'p_min': 0.02,
    'p_max': 0.2,
    'q_ratio': 1.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.quantum.error_correction_threshold")]:
    del sys.modules[name]
main = importlib.import_module("problems.quantum.error_correction_threshold.main")
problem_dir = Path("problems/quantum/error_correction_threshold")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
